# FoDS Assignment 1 — Part 1
## Polynomial Regression: Closed Form vs. Gradient Descent

This notebook completes **P1 only**. It loads the provided `poly_train.csv` without generating or modifying data, fits degrees 1–9 by least squares, implements Batch/SGD/Mini-batch GD from scratch for degree 9, studies learning rates, and verifies the GD weights against the closed-form solution.

**Data location:** the loader checks `data/poly_train.csv`, `data/poly train.csv`, and the equivalent names beside the notebook or one directory above it. Keep the supplied CSV unchanged.


## 0. Imports and configuration

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

SEED = 42  # used for the per-epoch shuffling in SGD and Mini-batch GD
OUT = Path("outputs") / "part1"
OUT.mkdir(parents=True, exist_ok=True)

np.set_printoptions(precision=6, suppress=False, linewidth=140)


## 1. Load the provided training data

In [ ]:
def find_train_csv():
    """Find the supplied training CSV without creating or altering any data."""
    filenames = ("poly_train.csv", "poly train.csv")
    search_dirs = (Path.cwd() / "data", Path.cwd(), Path.cwd().parent / "data", Path.cwd().parent)

    checked = []
    for directory in search_dirs:
        for filename in filenames:
            candidate = directory / filename
            checked.append(str(candidate))
            if candidate.is_file():
                return candidate

    checked_text = "\n".join(f"  - {path}" for path in checked)
    raise FileNotFoundError(
        "Could not find the provided poly_train.csv. Put the original CSV in a 'data' folder "
        "or beside this notebook. Paths checked:\n" + checked_text
    )


def load_xy_csv(path):
    """Load the CSV's x and y columns by name."""
    data = np.genfromtxt(path, delimiter=",", names=True, dtype=float, encoding="utf-8")
    if data.dtype.names is None:
        raise ValueError(f"Expected a CSV with column headers x,y: {path}")

    names = {name.strip().lower(): name for name in data.dtype.names}
    if "x" not in names or "y" not in names:
        raise ValueError(f"Expected columns named x and y; found {data.dtype.names} in {path}")

    x_values = np.atleast_1d(data[names["x"]]).astype(float)
    y_values = np.atleast_1d(data[names["y"]]).astype(float)
    if x_values.shape != y_values.shape:
        raise ValueError("The x and y columns must have the same length.")
    return x_values, y_values


TRAIN_PATH = find_train_csv()
x, t = load_xy_csv(TRAIN_PATH)

if len(x) != 10:
    raise ValueError(f"P1 expects the provided N=10 training set; loaded N={len(x)}.")

print(f"Loaded: {TRAIN_PATH}")
print(f"Number of training points: {len(x)}")
print(f"Random seed for shuffling: {SEED}")


## 2. Design matrix and objective

For degree \(d\), the columns of \(\Phi\) are \(1,x,\ldots,x^d\). The objective is the **unnormalised SSE** \(E(w)=
rac12\|\Phi w-t\|_2^2\); do not divide the gradient by the number of points.

In [ ]:
def design_matrix(x_values, degree):
    """Return the monomial design matrix [1, x, ..., x**degree]."""
    x_values = np.asarray(x_values, dtype=float).reshape(-1)
    return np.vander(x_values, N=degree + 1, increasing=True)


def polynomial_predict(x_values, weights):
    """Predict using weights in the original monomial basis."""
    x_values = np.asarray(x_values, dtype=float).reshape(-1)
    return design_matrix(x_values, len(weights) - 1) @ weights


def sse_loss(Phi, target, weights):
    """E(w) = 1/2 ||Phi w - target||^2."""
    residual = Phi @ weights - target
    return 0.5 * float(residual @ residual)


x_dense = np.linspace(0.0, 1.0, 400)
true_y_dense = np.sin(2 * np.pi * x_dense)  # plotting reference only; never used for fitting


## 3. Closed-form fits for degrees 1–9

Use `np.linalg.lstsq`, an SVD-based least-squares solver, rather than explicitly inverting \(\Phi^T\Phi\).

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 11), sharex=True)
closed_form_weights = {}

for degree, ax in zip(range(1, 10), axes.flat):
    Phi_d = design_matrix(x, degree)
    w_d = np.linalg.lstsq(Phi_d, t, rcond=None)[0]
    closed_form_weights[degree] = w_d

    ax.scatter(x, t, label="Training points", zorder=3)
    ax.plot(x_dense, polynomial_predict(x_dense, w_d), label=f"Degree {degree} fit")
    ax.plot(x_dense, true_y_dense, linestyle="--", label="True sin(2πx)")
    ax.set_title(f"Polynomial degree {degree}")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.grid(alpha=0.25)
    ax.legend(fontsize=7)

fig.suptitle("Closed-form polynomial regression (degrees 1–9)", y=1.01)
fig.tight_layout()
fig.savefig(OUT / "closed_form_fits.png", dpi=160, bbox_inches="tight")
plt.show()


## 4. QR reparameterisation for degree 9

The monomial design matrix for degree 9 is ill-conditioned. Compute a thin QR factorisation \(\Phi=QR\), set \(\alpha=Rw\), and perform all gradient updates in the stable \(\alpha\)-space objective \(E(\alpha)=\frac12\|Q\alpha-t\|_2^2\). Recover the original polynomial weights with \(w=R^{-1}\alpha\).

In [ ]:
DEGREE = 9
Phi9 = design_matrix(x, DEGREE)
Q, R = np.linalg.qr(Phi9, mode="reduced")

print(f"cond(Phi) for degree 9: {np.linalg.cond(Phi9):.6e}")
print(f"cond(Q) after QR:       {np.linalg.cond(Q):.6e}")


def alpha_loss(Q_matrix, target, alpha):
    """Equivalent objective in QR coordinates: 1/2 ||Q alpha - target||^2."""
    residual = Q_matrix @ alpha - target
    return 0.5 * float(residual @ residual)


def batch_gd(Q_matrix, target, eta, epochs, alpha0=None):
    """Batch GD: one update per epoch, using the full (unnormalised) gradient."""
    n_features = Q_matrix.shape[1]
    alpha = np.zeros(n_features) if alpha0 is None else np.asarray(alpha0, dtype=float).copy()
    losses = []

    for _ in range(epochs):
        gradient = Q_matrix.T @ (Q_matrix @ alpha - target)
        alpha -= eta * gradient
        losses.append(alpha_loss(Q_matrix, target, alpha))

    return alpha, np.asarray(losses)


def stochastic_gd(Q_matrix, target, eta, epochs, seed=SEED, alpha0=None):
    """SGD: shuffle the N samples each epoch and update using one sample at a time."""
    n_samples = len(target)
    n_features = Q_matrix.shape[1]
    alpha = np.zeros(n_features) if alpha0 is None else np.asarray(alpha0, dtype=float).copy()
    rng = np.random.default_rng(seed)
    losses = []

    for _ in range(epochs):
        for i in rng.permutation(n_samples):
            row = Q_matrix[i]
            sample_gradient = row * (row @ alpha - target[i])
            alpha -= eta * sample_gradient
        losses.append(alpha_loss(Q_matrix, target, alpha))

    return alpha, np.asarray(losses)


def minibatch_gd(Q_matrix, target, eta, epochs, batch_size=3, seed=SEED, alpha0=None):
    """Mini-batch GD: average each batch gradient; the final batch may be smaller."""
    n_samples = len(target)
    n_features = Q_matrix.shape[1]
    alpha = np.zeros(n_features) if alpha0 is None else np.asarray(alpha0, dtype=float).copy()
    rng = np.random.default_rng(seed)
    losses = []

    for _ in range(epochs):
        order = rng.permutation(n_samples)
        for start in range(0, n_samples, batch_size):
            idx = order[start:start + batch_size]
            Q_batch = Q_matrix[idx]
            residual = Q_batch @ alpha - target[idx]
            batch_gradient = (Q_batch.T @ residual) / len(idx)
            alpha -= eta * batch_gradient
        losses.append(alpha_loss(Q_matrix, target, alpha))

    return alpha, np.asarray(losses)


def alpha_to_weights(alpha):
    """Map QR coordinates back to the original polynomial coefficients."""
    return np.linalg.solve(R, alpha)


## 5. Loss curves: Batch GD, SGD, and Mini-batch GD

Use the same \(\eta=0.1\) for all methods. Each recorded point is the objective at the end of an epoch. SGD and Mini-batch GD shuffle with fixed seed `42`; the mini-batch size is 3, giving batch sizes 3, 3, 3, and 1 per epoch.

In [ ]:
ETA = 0.1
EPOCHS = 2000
alpha0 = np.zeros(Phi9.shape[1])

alpha_batch, losses_batch = batch_gd(Q, t, ETA, EPOCHS, alpha0)
alpha_sgd, losses_sgd = stochastic_gd(Q, t, ETA, EPOCHS, seed=SEED, alpha0=alpha0)
alpha_minibatch, losses_minibatch = minibatch_gd(
    Q, t, ETA, EPOCHS, batch_size=3, seed=SEED, alpha0=alpha0
)

# Protect the log plot if a loss rounds all the way to floating-point zero.
def positive_for_log(values):
    return np.maximum(np.asarray(values), np.finfo(float).tiny)

plt.figure(figsize=(9, 5.5))
plt.plot(np.arange(1, EPOCHS + 1), positive_for_log(losses_batch), label="Batch GD")
plt.plot(np.arange(1, EPOCHS + 1), positive_for_log(losses_sgd), label="SGD")
plt.plot(np.arange(1, EPOCHS + 1), positive_for_log(losses_minibatch), label="Mini-batch GD (size 3)")
plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("Loss E")
plt.title(f"Loss vs. epoch (η = {ETA})")
plt.grid(alpha=0.25, which="both")
plt.legend()
plt.tight_layout()
plt.savefig(OUT / "loss_curves.png", dpi=160, bbox_inches="tight")
plt.show()

print(f"Final Batch GD loss:       {losses_batch[-1]:.6e}")
print(f"Final SGD loss:            {losses_sgd[-1]:.6e}")
print(f"Final Mini-batch GD loss:  {losses_minibatch[-1]:.6e}")


## 6. Learning-rate study for Batch GD

Compare a small learning rate (`0.01`), a well-chosen learning rate (`0.5`), and a too-large learning rate (`2.1`). Use 100 epochs so that slow convergence and divergence are visible. The assignment also asks for each original-space weight vector \(w\) and its objective after exactly 100 epochs.

In [ ]:
LR_EPOCHS = 100
learning_rates = {
    "Too small (η = 0.01)": 0.01,
    "Well chosen (η = 0.5)": 0.5,
    "Too large (η = 2.1)": 2.1,
}

learning_rate_results = {}
plt.figure(figsize=(9, 5.5))

for label, eta in learning_rates.items():
    alpha_lr, losses_lr = batch_gd(Q, t, eta, LR_EPOCHS, np.zeros(Phi9.shape[1]))
    learning_rate_results[eta] = (alpha_lr, losses_lr)
    plt.plot(
        np.arange(1, LR_EPOCHS + 1),
        positive_for_log(losses_lr),
        label=label,
    )

plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("Loss E")
plt.title("Batch GD learning-rate study (degree 9, QR coordinates)")
plt.grid(alpha=0.25, which="both")
plt.legend()
plt.tight_layout()
plt.savefig(OUT / "learning_rate_study.png", dpi=160, bbox_inches="tight")
plt.show()


In [ ]:
print("Weights and loss after exactly 100 epochs (loss evaluated in equivalent QR coordinates):")
for eta in learning_rates.values():
    alpha_lr, losses_lr = learning_rate_results[eta]
    w_lr = alpha_to_weights(alpha_lr)
    final_loss = alpha_loss(Q, t, alpha_lr)
    print(f"\nη = {eta}")
    print("w =", np.array2string(w_lr, precision=6, suppress_small=False, max_line_width=140))
    print(f"E = {final_loss:.12e}")

print("\nAt exactly η = 2, the error component in each orthogonal QR direction changes sign but does not shrink;")
print("the loss therefore oscillates rather than converging (up to floating-point effects).")


## 7. Verification against the closed-form solution

Use the same fixed number of epochs for all methods, then compare their original-space weights to the SVD-based least-squares solution. Report both absolute and relative Euclidean errors.

In [ ]:
VERIFY_EPOCHS = 2000

# SVD-based closed-form least-squares solution in the original monomial basis.
w_closed = np.linalg.lstsq(Phi9, t, rcond=None)[0]

alpha_batch_v, _ = batch_gd(Q, t, ETA, VERIFY_EPOCHS, np.zeros(Phi9.shape[1]))
alpha_sgd_v, _ = stochastic_gd(Q, t, ETA, VERIFY_EPOCHS, seed=SEED, alpha0=np.zeros(Phi9.shape[1]))
alpha_minibatch_v, _ = minibatch_gd(
    Q, t, ETA, VERIFY_EPOCHS, batch_size=3, seed=SEED, alpha0=np.zeros(Phi9.shape[1])
)

verification_weights = {
    "Batch GD": alpha_to_weights(alpha_batch_v),
    "SGD": alpha_to_weights(alpha_sgd_v),
    "Mini-batch GD": alpha_to_weights(alpha_minibatch_v),
}

print(f"Closed-form weight norm ||w_closed||₂ = {np.linalg.norm(w_closed):.6e}")
print(f"Epochs per GD variant = {VERIFY_EPOCHS}; η = {ETA}; shuffle seed = {SEED}")
for name, w_gd in verification_weights.items():
    absolute_error = np.linalg.norm(w_gd - w_closed)
    relative_error = absolute_error / np.linalg.norm(w_closed)
    print(f"\n{name}")
    print(f"  Absolute error ||w_GD - w_closed||₂ = {absolute_error:.6e}")
    print(f"  Relative error                     = {relative_error:.6e}")


## 8. Discussion (for the report)

Refer to the loss-curve figure when writing this discussion. In this problem, the degree-9 design matrix is square, so the QR matrix \(Q\) is square and orthogonal: its rows are orthonormal. Consequently, one SGD pass updates each orthogonal direction once, making its end-of-epoch loss coincide with Batch GD's loss (apart from floating-point round-off), independent of the order of the shuffled samples. Mini-batch GD converges more slowly because its gradient is **averaged** within each batch, reducing the update scale for batches of size 3; the final batch has one sample as specified. The learning-rate figure should show slow progress for \(\eta=0.01\), rapid stable convergence for \(\eta=0.5\), and divergence for \(\eta=2.1\).